In [ ]:
# # Run AdsPower in headless mode with API key and custom API port
# adspower --headless=true --api-key=45b2f684c84679a888523e55a01b563b --api-port=50325 --no-sandbox

# # Run AdsPower without sandboxing
# adspower --no-sandbox

# # Run AdsPower in headless mode with Xvfb (X virtual framebuffer) for systems without a display
# xvfb-run adspower --headless=true --api-key=45b2f684c84679a888523e55a01b563b --api-port=50325 --no-sandbox


In [ ]:
import json
import requests
import subprocess

def create_group(group_name):
    url = "http://local.adspower.net:50325/api/v1/group/create"
    payload = {"group_name": group_name}
    headers = {'Content-Type': 'application/json'}
    response = requests.post(url, headers=headers, json=payload)
    return response.json()

def list_groups():
    url = "http://local.adspower.net:50325/api/v1/group/list?page=1&page_size=15"
    response = requests.get(url)
    return response.json()

def set_timezone_jakarta():
    try:
        subprocess.run(['sudo', 'timedatectl', 'set-timezone', 'Asia/Jakarta'], check=True)
        return "Timezone successfully set to Asia/Jakarta."
    except subprocess.CalledProcessError as e:
        return f"Error occurred while setting timezone: {e}"

def list_users():
    url = "http://local.adspower.net:50325/api/v1/user/list?page=1&page_size=100"
    headers = {}
    response = requests.request("GET", url, headers=headers)
    response_data = response.json()
    users = [{"name": user["name"], "user_id": user["user_id"]} for user in response_data["data"]["list"]]
    return users



In [20]:
import json
import requests
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import sys

from profile_class import Profile
from fingerprint_config import FingerprintConfig, RandomUA
from user_proxy_config import UserProxyConfig

def create_profile(payload):
    url = "http://local.adspower.net:50325/api/v1/user/create"
    headers = {
        'Content-Type': 'application/json'
    }
    
    response = requests.post(url, headers=headers, json=payload)
    return response.text

def start_browser_session(user_id):
    base_url = "http://local.adspower.net:50325/api/v1"
    open_url = f"{base_url}/browser/start?user_id={user_id}"
    close_url = f"{base_url}/browser/stop?user_id={user_id}"

    resp = requests.get(open_url).json()
    print(resp)
    if resp["code"] != 0:
        print(resp["msg"])
        print("Please check user_id")
        sys.exit()

    chrome_driver = resp["data"]["webdriver"]
    service = Service(executable_path=chrome_driver)
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", resp["data"]["ws"]["selenium"])

    driver = webdriver.Chrome(service=service, options=chrome_options)
    print(driver.title)
    
    return driver, close_url

def stop_browser_session(close_url):
    resp = requests.get(close_url).json()
    if resp["code"] == 0:
        print("Browser session stopped successfully.")
    else:
        print(f"Error stopping browser session: {resp['msg']}")

def perform_signup(driver, close_url):
    def highlight(element):
        driver.execute_script("arguments[0].style.border='3px solid red'", element)

    try:
        # Open the signup page
        driver.get("https://dashboard.residential.rayobyte.com/user-area/#/signup")
        # driver.refresh()
        time.sleep(3)

        # Wait for the page to load and locate input elements
        email_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//*[@id='eml']"))
        )
        pwd_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//*[@id='pwd']"))
        )

        # Highlight and enter the email and password
        highlight(email_input)
        highlight(pwd_input)
        email_input.send_keys("royaol@edgesoftware.xyz")
        time.sleep(2)
        pwd_input.send_keys("sBpCMKDIUU4A")

        # Locate, highlight, and click the checkbox
        checkbox = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//*[@id='input-47']"))
        )
        highlight(checkbox)
        checkbox.click()

        # Locate, highlight, and click the signup button
        signup_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//*[@id='app']/div/main/div/div[2]/div/div/form/div/div[2]/div/div/button/span"))
        )
        highlight(signup_btn)
        time.sleep(2)
        signup_btn.click()

        # Wait for and print the error message, if any
        try:
            error_message = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.XPATH, "//*[@id='app']/div[2]/div/div/div[2]/div/div/div/span"))
            )
            highlight(error_message)
            print("Error message content:", error_message.text)
        except Exception as e:
            print("Manual check required")
    except Exception as e:
        print("An error occurred during signup:", e)

    finally:
        # Let user decide to close the browser session
        # action = input("Press 'y' to close, press any other key to keep the browser session: ")
        action = 'y'
        if action.lower() == 'y':
            requests.get(close_url)
            stop_browser_session(close_url)
        else:
            print("Keeping the browser session open")

def delete_all_profiles():
    base_url = "http://local.adspower.net:50325"
    list_url = f"{base_url}/api/v1/user/list?page=1&page_size=100"
    delete_url = f"{base_url}/api/v1/user/delete"
    
    # Get the list of profiles
    response = requests.get(list_url)
    if response.status_code != 200:
        print(f"Error fetching profile list: {response.status_code}")
        return
    
    profiles_data = response.json()
    profiles = profiles_data.get("data", {}).get("list", [])
    user_ids = [profile['user_id'] for profile in profiles]
    
    if not user_ids:
        print("No profiles to delete.")
        return
    
    # Delete profiles
    payload = {"user_ids": user_ids}
    delete_response = requests.post(delete_url, headers={'Content-Type': 'application/json'}, json=payload)
    
    if delete_response.status_code == 200:
        print("All profiles deleted successfully.")
    else:
        print(f"Error deleting profiles: {delete_response.status_code} - {delete_response.text}")


In [ ]:
driver.get("https://dashboard.residential.rayobyte.com/user-area/#/signup")


In [ ]:
payload = Profile(
    name='test',
    group_id='4683840',
    fingerprint_config=FingerprintConfig(
        random_ua=RandomUA(ua_system_version=['Mac OS X 13'])
        # fonts=['all']
    ),
    # user_proxy_config=UserProxyConfig(proxy_soft='no_proxy')
    user_proxy_config=UserProxyConfig(proxy_soft='no_proxy')
).to_dict()

In [ ]:
id = 'kl3r0wu'
driver, close_url = start_browser_session(id)

In [21]:
for i in range(10):
    response = create_profile(payload)
    # Print the response and extract the user ID
    print(response)
    user_id = json.loads(response)["data"]["id"]
    print(f"User ID: {user_id}")
    driver, close_url = start_browser_session(user_id)
    perform_signup(driver, close_url)
    delete_all_profiles()

{"data":{"id":"kl3u782","serial_number":"75"},"code":0,"msg":"Success"}
User ID: kl3u782
{'code': 0, 'msg': 'success', 'data': {'ws': {'puppeteer': 'ws://127.0.0.1:40299/devtools/browser/62014ffc-1d05-4c55-883c-90596af7c109', 'selenium': '127.0.0.1:40299'}, 'debug_port': '40299', 'webdriver': '/root/.config/adspower_global/cwd_global/chrome_126/chromedriver'}}
75
An error occurred during signup: Message: 
Stacktrace:
#0 0x5648199d7312 <unknown>
#1 0x5648199c909e <unknown>
#2 0x564819681587 <unknown>
#3 0x5648196c9ddc <unknown>
#4 0x5648196ca171 <unknown>
#5 0x56481970cf74 <unknown>
#6 0x5648196ece6d <unknown>
#7 0x5648196beeaf <unknown>
#8 0x56481970ac5b <unknown>
#9 0x5648196ecae3 <unknown>
#10 0x5648196bdf31 <unknown>
#11 0x5648196bcfb9 <unknown>
#12 0x5648196bdd0c <unknown>
#13 0x5648199810ff <unknown>
#14 0x56481999ba75 <unknown>
#15 0x56481999b4e2 <unknown>
#16 0x56481999bf05 <unknown>
#17 0x56481998abf7 <unknown>
#18 0x56481999c2a0 <unknown>
#19 0x564819973101 <unknown>
#20 0x564

KeyboardInterrupt: 

In [ ]:
# Read the list of valid proxies
input_file = 'working_socks5_proxies.txt'

with open(input_file, 'r') as f:
    valid_proxies = [line.strip() for line in f]

# Generate UserProxyConfig objects for each proxy
user_proxy_configs = []
for proxy in valid_proxies:
    proxy_host, proxy_port = proxy.split(':')
    config = UserProxyConfig(
        proxy_soft="other",  # Assuming the proxy software is "other", change as needed
        proxy_type="http",
        proxy_host=proxy_host,
        proxy_port=proxy_port
    )
    user_proxy_configs.append(config)

# Print the generated UserProxyConfig objects
# for config in user_proxy_configs:
#     print(json.dumps(config.to_dict(), indent=4))

for user_proxy in user_proxy_configs:
    payload = Profile(
        name='test',
        group_id='4683840',
        fingerprint_config=FingerprintConfig(
            random_ua=RandomUA(ua_system_version=['Mac OS X 13'])
            # fonts=['all']
        ),
        user_proxy_config=user_proxy
    ).to_dict()
    # if after 10s no response, print error and go on to next one 
    response = create_profile(payload)
    
    
    print(response)
    user_id = json.loads(response)["data"]["id"]
    print(f"User ID: {user_id}")
    driver, close_url = start_browser_session(user_id)
    perform_signup(driver, close_url)
    delete_all_profiles()



In [18]:
# # Proxy settings
# proxy_host = "p.webshare.io"
# proxy_port = "80"
# proxy_username = "vmfblpzs-rotate"
# proxy_password = "u70jq9op8ef6"
payload = Profile(
    name='test',
    group_id='4683840',
    fingerprint_config=FingerprintConfig(
        random_ua=RandomUA(ua_system_version=['Mac OS X 13'])
        # fonts=['all']
    ),
    user_proxy_config=UserProxyConfig(proxy_soft = 'other', proxy_type = 'socks5', proxy_host = 'p.webshare.io', proxy_port = '80', proxy_user = 'vmfblpzs-rotate', proxy_password = 'u70jq9op8ef6')
).to_dict()

In [ ]:
payload = Profile(
    name='test',
    group_id='4683840',
    fingerprint_config=FingerprintConfig(
        random_ua=RandomUA(ua_system_version=['Mac OS X 13'])
        # fonts=['all']
    ),
    user_proxy_config=UserProxyConfig(proxy_soft='no_proxy')
).to_dict()